# Análisis de Redes Sociales: Estructura del Metro CDMX y Propagación de COVID-19 (2020-2021)

**Proyecto Completo:** Análisis de cómo la topología de la red del Metro de la Ciudad de México explica la difusión espacial de COVID-19 entre alcaldías.

---

## Tabla de Contenidos
1. Carga e Inspección de Datos
2. Construcción de la Red del Metro
3. Análisis Estructural de la Red
4. Visualizaciones de la Red
5. Simulación SIR en Red Real
6. Comparación con Redes Sintéticas
7. Validación con Machine Learning
8. Exportación de Resultados

In [ ]:
import os
import sys
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from scipy import stats
from scipy.spatial.distance import cdist
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import mean_absolute_error, r2_score
from datetime import datetime, timedelta
import random

# Configuración
warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)
os.makedirs('resultados', exist_ok=True)
os.makedirs('reporte', exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

print("✓ Librerías importadas correctamente")
print(f"Fecha/Hora: {datetime.now()}")

## PASO 0: Carga e Inspección de Datos

In [ ]:
# Cargar datos
try:
    df_metro = pd.read_csv('afluenciastc_2020_2021.csv', encoding='utf-8')
    df_covid = pd.read_csv('COVID19MEXICO2020_2021_CDMX.csv', encoding='utf-8')
    df_coords = pd.read_csv('estaciones_coords.csv', encoding='utf-8')
    print("✓ Archivos CSV cargados exitosamente\n")
except Exception as e:
    print(f"✗ Error al cargar CSV: {e}")
    sys.exit(1)

# Inspeccionar Metro data
print("=" * 80)
print("DATOS DE AFLUENCIA DEL METRO (2020-2021)")
print("=" * 80)
print(f"Shape: {df_metro.shape}")
print(f"Columnas: {list(df_metro.columns)}")
print(f"Tipos de datos:\n{df_metro.dtypes}")
print(f"\nPrimeras filas:\n{df_metro.head()}")
print(f"\nÚltimas filas:\n{df_metro.tail()}")
print(f"Valores únicos por columna:")
for col in df_metro.columns:
    print(f"  {col}: {df_metro[col].nunique()} únicos")

# Inspeccionar COVID data
print("\n" + "=" * 80)
print("DATOS DE COVID-19 (CDMX, 2020-2021)")
print("=" * 80)
print(f"Shape: {df_covid.shape}")
print(f"Columnas: {list(df_covid.columns)}")
print(f"Tipos de datos:\n{df_covid.dtypes}")
print(f"\nPrimeras filas:\n{df_covid.head()}")
print(f"\nÚltimas filas:\n{df_covid.tail()}")

# Inspeccionar Coordinates data
print("\n" + "=" * 80)
print("DATOS DE COORDENADAS DE ESTACIONES")
print("=" * 80)
print(f"Shape: {df_coords.shape}")
print(f"Columnas: {list(df_coords.columns)}")
print(f"Tipos de datos:\n{df_coords.dtypes}")
print(f"\nPrimeras filas:\n{df_coords.head()}")
print(f"\nÚltimas filas:\n{df_coords.tail()}")

In [ ]:
# Preparar datos para construcción de red
df_metro['fecha'] = pd.to_datetime(df_metro['fecha'])
df_metro['anio'] = df_metro['anio'].astype(int)

# Filtrar año 2020 para pesos de aristas
df_metro_2020 = df_metro[df_metro['anio'] == 2020].copy()

# Estandarizar nombres de estaciones (remover caracteres especiales)
df_metro['estacion'] = df_metro['estacion'].str.strip()
df_coords['estacion'] = df_coords['estacion'].str.strip()

# Información de estaciones
print(f"\nEstaciones únicas en Metro: {df_metro['estacion'].nunique()}")
print(f"Estaciones en Coordenadas: {df_coords.shape[0]}")
print(f"Líneas únicas: {df_metro['linea'].nunique()}")

# Crear diccionario estación -> alcaldía (MUNICIPIO_RES en COVID es alcaldía)
alcaldias_map = {
    1: 'Cuauhtémoc', 2: 'Miguel Hidalgo', 3: 'Venustiano Carranza',
    4: 'Benito Juárez', 5: 'Coyoacán', 6: 'Iztapalapa', 7: 'Gustavo A. Madero',
    8: 'Álvaro Obregón', 9: 'Iztacalco', 10: 'La Magdalena Contreras',
    11: 'Milpa Alta', 12: 'Tláhuac', 13: 'Tlalpan', 14: 'Xochimilco',
    15: 'Azcapotzalco', 16: 'Coyoacán', 17: 'Tláhuac'
}

print("\n✓ Datos preparados para construcción de red")

## PASO 1: Construcción de la Red del Metro

In [ ]:
# Construir red del Metro
G = nx.Graph()

# Obtener afluencia promedio por estación en 2020
afluencia_promedio = df_metro_2020.groupby('estacion')['afluencia'].mean()

# Crear nodos (estaciones)
estaciones_coords_dict = {}
for _, row in df_coords.iterrows():
    estacion = row['estacion']
    G.add_node(estacion, lat=row['lat'], lng=row['lng'])
    estaciones_coords_dict[estacion] = (row['lat'], row['lng'])

# Crear aristas entre estaciones adyacentes en la misma línea
print("Creando aristas entre estaciones adyacentes...")
aristas_por_linea = {}

for linea in df_metro_2020['linea'].unique():
    estaciones_linea = df_metro_2020[df_metro_2020['linea'] == linea].groupby('fecha')['estacion'].apply(list)
    
    # Crear aristas entre estaciones consecutivas en la línea
    for fecha, estaciones in estaciones_linea.items():
        for i in range(len(estaciones) - 1):
            u, v = estaciones[i], estaciones[i+1]
            
            # Verificar que ambas estaciones existan en el grafo
            if u in G and v in G:
                # Obtener afluencia promedio de ambas estaciones
                weight_u = afluencia_promedio.get(u, 0)
                weight_v = afluencia_promedio.get(v, 0)
                avg_weight = (weight_u + weight_v) / 2
                
                # Agregar o actualizar arista
                if G.has_edge(u, v):
                    # Actualizar peso (promediar)
                    G[u][v]['weight'] = (G[u][v]['weight'] + avg_weight) / 2
                else:
                    G.add_edge(u, v, weight=avg_weight, linea=linea)
                    if linea not in aristas_por_linea:
                        aristas_por_linea[linea] = 0
                    aristas_por_linea[linea] += 1

# Información de la red
print(f"\n✓ Red construida:")
print(f"  Nodos (estaciones): {G.number_of_nodes()}")
print(f"  Aristas (conexiones): {G.number_of_edges()}")
print(f"  Aristas por línea: {aristas_por_linea}")
print(f"  Densidad: {nx.density(G):.4f}")

# Verificar conectividad
if nx.is_connected(G):
    print(f"  Red: CONECTADA")
else:
    print(f"  Red: NO CONECTADA")
    componentes = list(nx.connected_components(G))
    print(f"  Componentes conexas: {len(componentes)}")
    for i, comp in enumerate(componentes[:3]):
        print(f"    Componente {i+1}: {len(comp)} nodos")

# Guardar red
nx.write_graphml(G, 'resultados/metro_network.graphml')
print(f"\n✓ Red guardada en: resultados/metro_network.graphml")

## PASO 2: Análisis Estructural de la Red

In [ ]:
# Calcular métricas de nodo
print("Calculando métricas de centralidad...")

node_metrics = {}

# Grado y grado ponderado
degree = dict(G.degree())
weighted_degree = dict(G.degree(weight='weight'))

# Betweenness centrality
betweenness = nx.betweenness_centrality(G, weight='weight')

# Closeness centrality
closeness = nx.closeness_centrality(G, distance='weight')

# Clustering coefficient
clustering = nx.clustering(G)

# Agregar todas las métricas
for nodo in G.nodes():
    node_metrics[nodo] = {
        'grado': degree[nodo],
        'grado_ponderado': weighted_degree[nodo],
        'betweenness': betweenness[nodo],
        'closeness': closeness[nodo],
        'clustering': clustering[nodo],
        'lat': G.nodes[nodo].get('lat', None),
        'lng': G.nodes[nodo].get('lng', None)
    }

# Convertir a DataFrame
df_metrics = pd.DataFrame(node_metrics).T.reset_index()
df_metrics.columns = ['estacion', 'grado', 'grado_ponderado', 'betweenness', 'closeness', 'clustering', 'lat', 'lng']

# Guardar métricas de nodo
df_metrics.to_csv('resultados/node_metrics.csv', index=False)
print(f"✓ Métricas de nodo guardadas en: resultados/node_metrics.csv")

# Métricas globales
print("\nMétricas Globales de la Red:")
print("=" * 50)

# Diámetro (solo si la red es conexa)
if nx.is_connected(G):
    diametro = nx.diameter(G)
    promedio_camino = nx.average_shortest_path_length(G)
    print(f"Diámetro: {diametro}")
    print(f"Longitud promedio del camino: {promedio_camino:.4f}")
else:
    # Para redes desconectadas, usar la componente más grande
    largest_cc = max(nx.connected_components(G), key=len)
    G_temp = G.subgraph(largest_cc)
    diametro = nx.diameter(G_temp)
    promedio_camino = nx.average_shortest_path_length(G_temp)
    print(f"Diámetro (componente principal): {diametro}")
    print(f"Longitud promedio del camino (componente principal): {promedio_camino:.4f}")

# Clustering global
clustering_global = nx.average_clustering(G)
print(f"Coeficiente de clustering global: {clustering_global:.4f}")

# Densidad
densidad = nx.density(G)
print(f"Densidad: {densidad:.4f}")

# Grado promedio
grado_promedio = sum(degree.values()) / len(degree)
print(f"Grado promedio: {grado_promedio:.4f}")

# Análisis de distribución de grado
grados = list(degree.values())
print(f"\nDistribución de grado:")
print(f"  Mínimo: {min(grados)}")
print(f"  Máximo: {max(grados)}")
print(f"  Media: {np.mean(grados):.4f}")
print(f"  Mediana: {np.median(grados):.4f}")
print(f"  Desv. Est.: {np.std(grados):.4f}")

## PASO 3: Visualizaciones de la Red

In [ ]:
# 1. Red geográfica
print("Creando visualización 1: Red Geográfica...")
fig, ax = plt.subplots(figsize=(14, 10), dpi=300)

# Extraer coordenadas
pos = {nodo: (G.nodes[nodo]['lng'], G.nodes[nodo]['lat']) for nodo in G.nodes()}

# Normalizar métricas para tamaño y color
betweenness_vals = np.array([betweenness[n] for n in G.nodes()])
weighted_degree_vals = np.array([weighted_degree[n] for n in G.nodes()])

# Normalizar a rango [300, 3000] para tamaño
size_nodes = 300 + 2700 * (betweenness_vals - betweenness_vals.min()) / (betweenness_vals.max() - betweenness_vals.min() + 1e-9)

# Dibujar aristas
nx.draw_networkx_edges(G, pos, alpha=0.2, width=0.5, edge_color='gray', ax=ax)

# Dibujar nodos
nodes = nx.draw_networkx_nodes(
    G, pos, 
    node_size=size_nodes,
    node_color=weighted_degree_vals,
    cmap='YlOrRd',
    alpha=0.8,
    ax=ax
)

# Etiquetas para los nodos principales (top 10)
top_10_nodes = df_metrics.nlargest(10, 'betweenness')['estacion'].tolist()
labels = {n: n if n in top_10_nodes else '' for n in G.nodes()}
nx.draw_networkx_labels(G, pos, labels, font_size=6, ax=ax)

ax.set_title('Red del Metro CDMX: Topología Geográfica\n(Tamaño = Betweenness, Color = Grado Ponderado)', fontsize=14, fontweight='bold')
ax.set_xlabel('Longitud', fontsize=11)
ax.set_ylabel('Latitud', fontsize=11)
cbar = plt.colorbar(nodes, ax=ax, label='Grado Ponderado (Afluencia)')
ax.axis('off')
plt.tight_layout()
plt.savefig('resultados/red_metro_geografica.png', dpi=300, bbox_inches='tight')
print("✓ Guardado: red_metro_geografica.png")
plt.close()

# 2. Distribución de grado (log-log)
print("Creando visualización 2: Distribución de Grado...")
fig, ax = plt.subplots(figsize=(10, 7), dpi=300)

grados_count = pd.Series(grados).value_counts().sort_index()
ax.loglog(grados_count.index, grados_count.values, 'o-', linewidth=2, markersize=8, color='darkblue')
ax.set_xlabel('Grado (log)', fontsize=12)
ax.set_ylabel('Frecuencia (log)', fontsize=12)
ax.set_title('Distribución de Grado - Red del Metro CDMX', fontsize=14, fontweight='bold')
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.savefig('resultados/distribucion_grado.png', dpi=300, bbox_inches='tight')
print("✓ Guardado: distribucion_grado.png")
plt.close()

# 3. Top 10 estaciones por betweenness
print("Creando visualización 3: Top 10 Estaciones...")
top_10 = df_metrics.nlargest(10, 'betweenness')[['estacion', 'betweenness']]
fig, ax = plt.subplots(figsize=(10, 7), dpi=300)

ax.barh(range(len(top_10)), top_10['betweenness'].values, color='steelblue')
ax.set_yticks(range(len(top_10)))
ax.set_yticklabels(top_10['estacion'].values, fontsize=10)
ax.set_xlabel('Centralidad de Intermediación (Betweenness)', fontsize=12)
ax.set_title('Top 10 Estaciones del Metro por Centralidad de Intermediación', fontsize=14, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('resultados/top10_estaciones.png', dpi=300, bbox_inches='tight')
print("✓ Guardado: top10_estaciones.png")
plt.close()

print("\n✓ Todas las visualizaciones completadas")

## PASO 4: Simulación SIR en Red Real

In [ ]:
# Función para simular SIR en una red
def simulate_sir(graph, initial_nodes, beta=0.3, gamma=0.1, max_time=200):
    """
    Simular modelo SIR en un grafo
    S: Susceptible, I: Infectado, R: Recuperado
    """
    n = graph.number_of_nodes()
    
    # Inicializar estados: 0=S, 1=I, 2=R
    state = np.zeros(n, dtype=int)
    node_list = list(graph.nodes())
    node_to_idx = {node: i for i, node in enumerate(node_list)}
    
    # Infectar nodos iniciales
    for node in initial_nodes:
        state[node_to_idx[node]] = 1
    
    # Registrar evolución
    susceptible = [np.sum(state == 0)]
    infected = [np.sum(state == 1)]
    recovered = [np.sum(state == 2)]
    
    # Simulación
    for t in range(max_time):
        new_state = state.copy()
        
        # Transmisión
        for node in graph.nodes():
            i = node_to_idx[node]
            if state[i] == 1:  # Si está infectado
                for neighbor in graph.neighbors(node):
                    j = node_to_idx[neighbor]
                    if state[j] == 0 and np.random.random() < beta:  # Susceptible
                        new_state[j] = 1
            
            # Recuperación
            if state[i] == 1 and np.random.random() < gamma:
                new_state[i] = 2
        
        state = new_state
        susceptible.append(np.sum(state == 0))
        infected.append(np.sum(state == 1))
        recovered.append(np.sum(state == 2))
        
        # Detener si no hay infectados
        if infected[-1] == 0:
            break
    
    return {
        'susceptible': susceptible,
        'infected': infected,
        'recovered': recovered,
        'peak_infected': max(infected),
        'peak_infected_pct': max(infected) / n * 100,
        'time_to_peak': infected.index(max(infected))
    }

print("Modelo SIR implementado")
print("Parámetros: beta=0.3, gamma=0.1\n")

# Top 3 estaciones por betweenness
top_3_betweenness = df_metrics.nlargest(3, 'betweenness')['estacion'].tolist()
print(f"Top 3 estaciones por betweenness:")
for i, estacion in enumerate(top_3_betweenness, 1):
    print(f"  {i}. {estacion}")

# Simulaciones desde estaciones de alto betweenness
print("\nEjecutando 5 simulaciones desde top-3 nodos...")
results_high_betweenness = []
for sim in range(5):
    np.random.seed(42 + sim)
    random.seed(42 + sim)
    result = simulate_sir(G, top_3_betweenness, beta=0.3, gamma=0.1, max_time=200)
    results_high_betweenness.append(result)
    print(f"  Sim {sim+1}: Peak={result['peak_infected_pct']:.1f}%, Time={result['time_to_peak']}")

# Simulaciones desde nodos aleatorios
print("\nEjecutando 5 simulaciones desde nodos aleatorios...")
random_nodes_list = [random.sample(list(G.nodes()), 3) for _ in range(5)]
results_random = []
for sim in range(5):
    np.random.seed(42 + 5 + sim)
    random.seed(42 + 5 + sim)
    result = simulate_sir(G, random_nodes_list[sim], beta=0.3, gamma=0.1, max_time=200)
    results_random.append(result)
    print(f"  Sim {sim+1}: Peak={result['peak_infected_pct']:.1f}%, Time={result['time_to_peak']}")

# Visualizar SIR en red real
print("\nCreando visualización SIR...")
fig, ax = plt.subplots(figsize=(12, 7), dpi=300)

# Plot de simulaciones desde high betweenness
for i, result in enumerate(results_high_betweenness):
    tiempo = range(len(result['infected']))
    infected_pct = np.array(result['infected']) / G.number_of_nodes() * 100
    ax.plot(tiempo, infected_pct, linewidth=2, alpha=0.7, label=f'High Betweenness Sim {i+1}', linestyle='-')

# Plot de simulaciones desde random
for i, result in enumerate(results_random):
    tiempo = range(len(result['infected']))
    infected_pct = np.array(result['infected']) / G.number_of_nodes() * 100
    ax.plot(tiempo, infected_pct, linewidth=2, alpha=0.7, label=f'Random Sim {i+1}', linestyle='--')

ax.set_xlabel('Tiempo (días)', fontsize=12)
ax.set_ylabel('Infectados (%)', fontsize=12)
ax.set_title('Simulación SIR en Red Real del Metro CDMX\n(β=0.3, γ=0.1)', fontsize=14, fontweight='bold')
ax.legend(fontsize=8, loc='best')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('resultados/simulacion_SIR_real.png', dpi=300, bbox_inches='tight')
print("✓ Guardado: simulacion_SIR_real.png")
plt.close()

print("\n✓ Simulaciones SIR completadas")

## PASO 5: Comparación con Redes Sintéticas

In [ ]:
# Parámetros de la red real
n_nodes = G.number_of_nodes()
n_edges = G.number_of_edges()
avg_degree = 2 * n_edges / n_nodes

print(f"Parámetros de la red real:")
print(f"  Nodos: {n_nodes}")
print(f"  Aristas: {n_edges}")
print(f"  Grado promedio: {avg_degree:.4f}\n")

# Generar redes sintéticas
print("Generando redes sintéticas...")

# 1. Erdős-Rényi (random)
p = 2 * n_edges / (n_nodes * (n_nodes - 1))
G_er = nx.erdos_renyi_graph(n_nodes, p, seed=42)
print(f"✓ Erdős-Rényi: {G_er.number_of_nodes()} nodos, {G_er.number_of_edges()} aristas")

# 2. Watts-Strogatz (small-world)
k = int(avg_degree)
G_ws = nx.watts_strogatz_graph(n_nodes, k, p=0.1, seed=42)
print(f"✓ Watts-Strogatz (k={k}, p=0.1): {G_ws.number_of_nodes()} nodos, {G_ws.number_of_edges()} aristas")

# 3. Barabási-Albert (scale-free)
m = 2
G_ba = nx.barabasi_albert_graph(n_nodes, m, seed=42)
print(f"✓ Barabási-Albert (m={m}): {G_ba.number_of_nodes()} nodos, {G_ba.number_of_edges()} aristas")

# Función para ejecutar todas las simulaciones en una red
def run_sir_simulations(graph, n_sims=5, initial_size=3):
    """Ejecutar múltiples simulaciones SIR en una red"""
    results = []
    for sim in range(n_sims):
        np.random.seed(42 + sim)
        random.seed(42 + sim)
        initial = random.sample(list(graph.nodes()), min(initial_size, graph.number_of_nodes()))
        result = simulate_sir(graph, initial, beta=0.3, gamma=0.1, max_time=200)
        results.append(result)
    return results

# Ejecutar simulaciones SIR en redes sintéticas
print("\nEjecutando simulaciones SIR en redes sintéticas...")
results_er = run_sir_simulations(G_er, n_sims=5)
print(f"  Erdős-Rényi - Peak promedio: {np.mean([r['peak_infected_pct'] for r in results_er]):.1f}%")

results_ws = run_sir_simulations(G_ws, n_sims=5)
print(f"  Watts-Strogatz - Peak promedio: {np.mean([r['peak_infected_pct'] for r in results_ws]):.1f}%")

results_ba = run_sir_simulations(G_ba, n_sims=5)
print(f"  Barabási-Albert - Peak promedio: {np.mean([r['peak_infected_pct'] for r in results_ba]):.1f}%")

# Comparación de SIR
print("\nComparación de picos de infección:")
peaks_real = [r['peak_infected_pct'] for r in results_high_betweenness + results_random]
peaks_er = [r['peak_infected_pct'] for r in results_er]
peaks_ws = [r['peak_infected_pct'] for r in results_ws]
peaks_ba = [r['peak_infected_pct'] for r in results_ba]

print(f"  Red Real: media={np.mean(peaks_real):.1f}%, std={np.std(peaks_real):.1f}%")
print(f"  ER: media={np.mean(peaks_er):.1f}%, std={np.std(peaks_er):.1f}%")
print(f"  WS: media={np.mean(peaks_ws):.1f}%, std={np.std(peaks_ws):.1f}%")
print(f"  BA: media={np.mean(peaks_ba):.1f}%, std={np.std(peaks_ba):.1f}%")

# Tests Mann-Whitney U
print("\nTests Mann-Whitney U (Real vs Sintéticas):")
u_er, p_er = stats.mannwhitneyu(peaks_real, peaks_er)
u_ws, p_ws = stats.mannwhitneyu(peaks_real, peaks_ws)
u_ba, p_ba = stats.mannwhitneyu(peaks_real, peaks_ba)
print(f"  Real vs ER: U={u_er:.1f}, p-value={p_er:.4f}")
print(f"  Real vs WS: U={u_ws:.1f}, p-value={p_ws:.4f}")
print(f"  Real vs BA: U={u_ba:.1f}, p-value={p_ba:.4f}")

# Visualizar comparación SIR
print("\nCreando visualización de comparación...")
fig, ax = plt.subplots(figsize=(12, 7), dpi=300)

# Plot de todas las simulaciones
# Real
for i, result in enumerate(results_high_betweenness):
    tiempo = range(len(result['infected']))
    infected_pct = np.array(result['infected']) / G.number_of_nodes() * 100
    ax.plot(tiempo, infected_pct, linewidth=1.5, alpha=0.6, color='red', linestyle='-')

# ER
for i, result in enumerate(results_er):
    tiempo = range(len(result['infected']))
    infected_pct = np.array(result['infected']) / G_er.number_of_nodes() * 100
    ax.plot(tiempo, infected_pct, linewidth=1.5, alpha=0.6, color='blue', linestyle='--')

# WS
for i, result in enumerate(results_ws):
    tiempo = range(len(result['infected']))
    infected_pct = np.array(result['infected']) / G_ws.number_of_nodes() * 100
    ax.plot(tiempo, infected_pct, linewidth=1.5, alpha=0.6, color='green', linestyle=':')

# BA
for i, result in enumerate(results_ba):
    tiempo = range(len(result['infected']))
    infected_pct = np.array(result['infected']) / G_ba.number_of_nodes() * 100
    ax.plot(tiempo, infected_pct, linewidth=1.5, alpha=0.6, color='orange', linestyle='-.')

# Leyendas
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='red', lw=2, linestyle='-', label='Red Real'),
    Line2D([0], [0], color='blue', lw=2, linestyle='--', label='Erdős-Rényi'),
    Line2D([0], [0], color='green', lw=2, linestyle=':', label='Watts-Strogatz'),
    Line2D([0], [0], color='orange', lw=2, linestyle='-.', label='Barabási-Albert')
]
ax.legend(handles=legend_elements, fontsize=11, loc='best')

ax.set_xlabel('Tiempo (días)', fontsize=12)
ax.set_ylabel('Infectados (%)', fontsize=12)
ax.set_title('Comparación SIR: Red Real vs Redes Sintéticas\n(β=0.3, γ=0.1)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('resultados/comparacion_SIR.png', dpi=300, bbox_inches='tight')
print("✓ Guardado: comparacion_SIR.png")
plt.close()

print("\n✓ Comparación de redes sintéticas completada")

## PASO 6: Validación con Machine Learning

In [ ]:
# Mapeo de estaciones a alcaldías (manual mapping basado en coordenadas)
# Diccionario de alcaldías con límites aproximados de lat/lon
alcaldias_bounds = {
    'Cuauhtémoc': {'lat_min': 19.390, 'lat_max': 19.435, 'lng_min': -99.150, 'lng_max': -99.120},
    'Miguel Hidalgo': {'lat_min': 19.400, 'lat_max': 19.490, 'lng_min': -99.230, 'lng_max': -99.140},
    'Venustiano Carranza': {'lat_min': 19.390, 'lat_max': 19.470, 'lng_min': -99.100, 'lng_max': -99.000},
    'Benito Juárez': {'lat_min': 19.330, 'lat_max': 19.420, 'lng_min': -99.200, 'lng_max': -99.130},
    'Coyoacán': {'lat_min': 19.300, 'lat_max': 19.380, 'lng_min': -99.250, 'lng_max': -99.140},
    'Iztapalapa': {'lat_min': 19.290, 'lat_max': 19.420, 'lng_min': -99.100, 'lng_max': -98.950},
    'Gustavo A. Madero': {'lat_min': 19.460, 'lat_max': 19.570, 'lng_min': -99.150, 'lng_max': -99.020},
    'Álvaro Obregón': {'lat_min': 19.300, 'lat_max': 19.450, 'lng_min': -99.350, 'lng_max': -99.200},
    'Iztacalco': {'lat_min': 19.370, 'lat_max': 19.430, 'lng_min': -99.100, 'lng_max': -99.020},
    'Tláhuac': {'lat_min': 19.240, 'lat_max': 19.360, 'lng_min': -99.100, 'lng_max': -98.950},
    'Tlalpan': {'lat_min': 19.200, 'lat_max': 19.330, 'lng_min': -99.250, 'lng_max': -99.050},
    'Xochimilco': {'lat_min': 19.250, 'lat_max': 19.330, 'lng_min': -99.200, 'lng_max': -99.030},
    'Azcapotzalco': {'lat_min': 19.450, 'lat_max': 19.570, 'lng_min': -99.350, 'lng_max': -99.200},
    'La Magdalena Contreras': {'lat_min': 19.250, 'lat_max': 19.380, 'lng_min': -99.350, 'lng_max': -99.240},
    'Milpa Alta': {'lat_min': 19.150, 'lat_max': 19.300, 'lng_min': -99.150, 'lng_max': -99.000},
}

# Mapear estaciones a alcaldías
estacion_alcaldia = {}
for _, row in df_metrics.iterrows():
    estacion = row['estacion']
    lat, lng = row['lat'], row['lng']
    
    alcaldia = None
    for alc, bounds in alcaldias_bounds.items():
        if (bounds['lat_min'] <= lat <= bounds['lat_max'] and 
            bounds['lng_min'] <= lng <= bounds['lng_max']):
            alcaldia = alc
            break
    
    estacion_alcaldia[estacion] = alcaldia if alcaldia else 'Otra'

print(f"Estaciones mapeadas a alcaldías: {len(estacion_alcaldia)}")
print(f"Alcaldías únicas: {len(set(estacion_alcaldia.values()))}")

# Agregar columna de alcaldía a métricas de nodo
df_metrics['alcaldia'] = df_metrics['estacion'].map(estacion_alcaldia)

# Agregar métricas de nodo por alcaldía
df_alcaldia_metrics = []
for alcaldia in df_metrics['alcaldia'].unique():
    if pd.isna(alcaldia):
        continue
    
    subset = df_metrics[df_metrics['alcaldia'] == alcaldia]
    
    metrics = {
        'alcaldia': alcaldia,
        'mean_betweenness': subset['betweenness'].mean(),
        'mean_grado': subset['grado'].mean(),
        'mean_clustering': subset['clustering'].mean(),
        'total_grado_ponderado': subset['grado_ponderado'].sum(),
        'n_estaciones': len(subset)
    }
    df_alcaldia_metrics.append(metrics)

df_alcaldia_metrics = pd.DataFrame(df_alcaldia_metrics)
print(f"\n✓ Métricas por alcaldía calculadas: {len(df_alcaldia_metrics)}")
print(df_alcaldia_metrics.head())

# Agregar casos COVID por alcaldía
# MUNICIPIO_RES = alcaldía en CDMX
df_covid['MUNICIPIO_RES'] = df_covid['MUNICIPIO_RES'].astype(int)

# Mapeo de código de municipio a nombre de alcaldía
municipio_map = {
    1: 'Cuauhtémoc', 2: 'Miguel Hidalgo', 3: 'Venustiano Carranza',
    4: 'Benito Juárez', 5: 'Coyoacán', 6: 'Iztapalapa', 7: 'Gustavo A. Madero',
    8: 'Álvaro Obregón', 9: 'Iztacalco', 10: 'La Magdalena Contreras',
    11: 'Milpa Alta', 12: 'Tláhuac', 13: 'Tlalpan', 14: 'Xochimilco',
    15: 'Azcapotzalco', 16: 'Coyoacán'  # Duplicado en original
}

df_covid['alcaldia'] = df_covid['MUNICIPIO_RES'].map(municipio_map)
df_covid['FECHA_INGRESO'] = pd.to_datetime(df_covid['FECHA_INGRESO'])

# Casos totales por alcaldía (2020-2021)
covid_alcaldia = df_covid.groupby('alcaldia').size().reset_index(name='total_casos')
print(f"\n✓ Casos COVID por alcaldía:")
print(covid_alcaldia)

# Merge de datos
df_ml = pd.merge(df_alcaldia_metrics, covid_alcaldia, on='alcaldia', how='inner')
print(f"\n✓ Datos merged para ML: {len(df_ml)} alcaldías")
print(df_ml)

# Preparar datos para ML
X = df_ml[['mean_betweenness', 'mean_grado', 'mean_clustering', 'total_grado_ponderado']].values
y = df_ml['total_casos'].values

print(f"\nDatos para ML:")
print(f"  X shape: {X.shape}")
print(f"  y shape: {y.shape}")
print(f"  Casos: media={y.mean():.0f}, min={y.min()}, max={y.max()}")

# Escalar features
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Leave-One-Out Cross-Validation con Random Forest
print("\nEntrenando Random Forest con Leave-One-Out CV...")
rf = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1)
loo = LeaveOneOut()

y_pred = np.zeros(len(y))
for train_idx, test_idx in loo.split(X_scaled):
    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    rf.fit(X_train, y_train)
    y_pred[test_idx] = rf.predict(X_test)

# Métricas
r2 = r2_score(y, y_pred)
mae = mean_absolute_error(y, y_pred)
rmse = np.sqrt(np.mean((y - y_pred) ** 2))

print(f"✓ Resultados Random Forest (LOO-CV):")
print(f"  R²: {r2:.4f}")
print(f"  MAE: {mae:.2f} casos")
print(f"  RMSE: {rmse:.2f} casos")

# Feature importance
rf.fit(X_scaled, y)  # Entrenar en todos los datos para feature importance
feature_names = ['Mean Betweenness', 'Mean Degree', 'Mean Clustering', 'Total Weighted Degree']
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]

print(f"\nImportancia de features:")
for i in indices:
    print(f"  {feature_names[i]}: {importances[i]:.4f}")

# Guardar modelo
import pickle
with open('resultados/modelo_rf.pkl', 'wb') as f:
    pickle.dump((rf, scaler), f)

print("\n✓ Validación ML completada")

In [ ]:
# Visualizaciones de ML
print("Creando visualizaciones de ML...")

# 1. Feature importances
fig, ax = plt.subplots(figsize=(10, 6), dpi=300)
colors = plt.cm.viridis(np.linspace(0, 1, len(indices)))
y_pos = np.arange(len(indices))
ax.barh(y_pos, importances[indices], color=colors)
ax.set_yticks(y_pos)
ax.set_yticklabels([feature_names[i] for i in indices], fontsize=11)
ax.set_xlabel('Importancia Relativa', fontsize=12)
ax.set_title('Importancia de Features - Random Forest\n(LOO-CV, predicción de casos COVID)', fontsize=14, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('resultados/feature_importances.png', dpi=300, bbox_inches='tight')
print("✓ Guardado: feature_importances.png")
plt.close()

# 2. Predicho vs Real
fig, ax = plt.subplots(figsize=(10, 8), dpi=300)
ax.scatter(y, y_pred, s=100, alpha=0.7, edgecolors='black', linewidth=1.5)

# Línea de identidad
min_val = min(y.min(), y_pred.min())
max_val = max(y.max(), y_pred.max())
ax.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Predicción perfecta')

# Anotaciones de alcaldías
for i, alcaldia in enumerate(df_ml['alcaldia']):
    ax.annotate(alcaldia, (y[i], y_pred[i]), fontsize=8, alpha=0.7, 
                xytext=(5, 5), textcoords='offset points')

ax.set_xlabel('Casos Reales', fontsize=12)
ax.set_ylabel('Casos Predichos', fontsize=12)
ax.set_title(f'Predicciones vs Valores Reales (R²={r2:.4f}, MAE={mae:.0f})', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('resultados/predicho_vs_real.png', dpi=300, bbox_inches='tight')
print("✓ Guardado: predicho_vs_real.png")
plt.close()

print("✓ Visualizaciones ML completadas")

## PASO 7: Exportación de Resultados

In [ ]:
# Exportar todos los datos
print("Exportando datos a resultados/...")

# CSV: métricas de nodo
df_metrics.to_csv('resultados/node_metrics.csv', index=False)
print("✓ node_metrics.csv")

# CSV: métricas por alcaldía
df_alcaldia_metrics.to_csv('resultados/alcaldia_metrics.csv', index=False)
print("✓ alcaldia_metrics.csv")

# CSV: datos ML
df_ml.to_csv('resultados/ml_data.csv', index=False)
print("✓ ml_data.csv")

# Crear resumen de métricas
resumen_metricas = {
    "proyecto": "Análisis de Redes Sociales: Metro CDMX y COVID-19",
    "fecha_generacion": datetime.now().isoformat(),
    "metricas_globales": {
        "nodos_estaciones": G.number_of_nodes(),
        "aristas_conexiones": G.number_of_edges(),
        "densidad": round(nx.density(G), 4),
        "diametro": int(diametro) if 'diametro' in locals() else None,
        "longitud_promedio_camino": round(float(promedio_camino), 4) if 'promedio_camino' in locals() else None,
        "coeficiente_clustering_global": round(clustering_global, 4),
        "grado_promedio": round(grado_promedio, 4)
    },
    "distribucion_grado": {
        "minimo": int(min(grados)),
        "maximo": int(max(grados)),
        "media": round(float(np.mean(grados)), 4),
        "mediana": round(float(np.median(grados)), 4),
        "desv_est": round(float(np.std(grados)), 4)
    },
    "top_10_estaciones": df_metrics.nlargest(10, 'betweenness')[['estacion', 'betweenness', 'grado_ponderado']].to_dict('records'),
    "simulaciones_sir": {
        "red_real": {
            "peak_infectados_promedio_pct": round(np.mean([r['peak_infected_pct'] for r in results_high_betweenness + results_random]), 2),
            "peak_infectados_std": round(np.std([r['peak_infected_pct'] for r in results_high_betweenness + results_random]), 2),
            "desde_high_betweenness_peak": [round(r['peak_infected_pct'], 2) for r in results_high_betweenness],
            "desde_random_peak": [round(r['peak_infected_pct'], 2) for r in results_random]
        },
        "erdos_renyi": {
            "peak_infectados_promedio_pct": round(np.mean([r['peak_infected_pct'] for r in results_er]), 2),
            "peak_infectados_std": round(np.std([r['peak_infected_pct'] for r in results_er]), 2)
        },
        "watts_strogatz": {
            "peak_infectados_promedio_pct": round(np.mean([r['peak_infected_pct'] for r in results_ws]), 2),
            "peak_infectados_std": round(np.std([r['peak_infected_pct'] for r in results_ws]), 2)
        },
        "barabasi_albert": {
            "peak_infectados_promedio_pct": round(np.mean([r['peak_infected_pct'] for r in results_ba]), 2),
            "peak_infectados_std": round(np.std([r['peak_infected_pct'] for r in results_ba]), 2)
        },
        "tests_mann_whitney": {
            "real_vs_erdos_renyi": {"U": round(float(u_er), 2), "p_value": round(float(p_er), 4)},
            "real_vs_watts_strogatz": {"U": round(float(u_ws), 2), "p_value": round(float(p_ws), 4)},
            "real_vs_barabasi_albert": {"U": round(float(u_ba), 2), "p_value": round(float(p_ba), 4)}
        }
    },
    "machine_learning": {
        "modelo": "Random Forest",
        "validacion_cruzada": "Leave-One-Out (LOO)",
        "features": feature_names,
        "r2_score": round(r2, 4),
        "mae": round(mae, 2),
        "rmse": round(rmse, 2),
        "feature_importances": {name: round(float(imp), 4) for name, imp in zip(feature_names, importances)},
        "n_alcaldias": len(df_ml),
        "casos_covid_total": int(y.sum())
    },
    "datos_procesados": {
        "estaciones_unicas": int(df_metro['estacion'].nunique()),
        "lineas_metro": int(df_metro['linea'].nunique()),
        "casos_covid_cdmx_2020_2021": int(len(df_covid)),
        "alcaldias_procesadas": len(df_alcaldia_metrics)
    }
}

# Guardar resumen
with open('resultados/resumen_metricas.json', 'w', encoding='utf-8') as f:
    json.dump(resumen_metricas, f, ensure_ascii=False, indent=2)
print("✓ resumen_metricas.json")

# Listar archivos generados
import glob
archivos = glob.glob('resultados/*')
print(f"\n✓ Archivos en resultados/:")
for archivo in sorted(archivos):
    size = os.path.getsize(archivo)
    size_str = f"{size/1024:.1f}KB" if size > 1024 else f"{size}B"
    print(f"  - {os.path.basename(archivo)} ({size_str})")

print("\n✓ Exportación completada")

## CHECKLIST FINAL

In [ ]:
print("=" * 80)
print("PROYECTO COMPLETADO - CHECKLIST FINAL")
print("=" * 80)

checklist = {
    "✓ PASO 0: Carga e Inspección de Datos": True,
    "✓ PASO 1: Construcción de Red del Metro": os.path.exists('resultados/metro_network.graphml'),
    "✓ PASO 2: Análisis Estructural": os.path.exists('resultados/node_metrics.csv'),
    "✓ PASO 3: Visualizaciones": (
        os.path.exists('resultados/red_metro_geografica.png') and
        os.path.exists('resultados/distribucion_grado.png') and
        os.path.exists('resultados/top10_estaciones.png')
    ),
    "✓ PASO 4: Simulación SIR Real": os.path.exists('resultados/simulacion_SIR_real.png'),
    "✓ PASO 5: Redes Sintéticas": os.path.exists('resultados/comparacion_SIR.png'),
    "✓ PASO 6: Machine Learning": (
        os.path.exists('resultados/feature_importances.png') and
        os.path.exists('resultados/predicho_vs_real.png')
    ),
    "✓ PASO 7: Exportación de Resultados": os.path.exists('resultados/resumen_metricas.json'),
}

for tarea, completada in checklist.items():
    estado = "✓ COMPLETADO" if completada else "✗ INCOMPLETO"
    print(f"{tarea}: {estado}")

print("\n" + "=" * 80)
print("RESUMEN DE RESULTADOS PRINCIPALES")
print("=" * 80)
print(f"\nRED DEL METRO:")
print(f"  - Estaciones (nodos): {G.number_of_nodes()}")
print(f"  - Conexiones (aristas): {G.number_of_edges()}")
print(f"  - Densidad: {nx.density(G):.6f}")
print(f"  - Grado promedio: {grado_promedio:.2f}")

print(f"\nMÉTRICAS GLOBALES:")
print(f"  - Coeficiente de clustering: {clustering_global:.4f}")
print(f"  - Diámetro: {diametro}")
print(f"  - Longitud promedio del camino: {promedio_camino:.4f}")

print(f"\nSIMULACIONES SIR (Red Real):")
print(f"  - Peak infectados (High Betweenness): {np.mean([r['peak_infected_pct'] for r in results_high_betweenness]):.1f}%")
print(f"  - Peak infectados (Random): {np.mean([r['peak_infected_pct'] for r in results_random]):.1f}%")

print(f"\nCOMPARACIÓN CON REDES SINTÉTICAS (Peak Infectados):")
print(f"  - Erdős-Rényi: {np.mean([r['peak_infected_pct'] for r in results_er]):.1f}%")
print(f"  - Watts-Strogatz: {np.mean([r['peak_infected_pct'] for r in results_ws]):.1f}%")
print(f"  - Barabási-Albert: {np.mean([r['peak_infected_pct'] for r in results_ba]):.1f}%")

print(f"\nMODELO MACHINE LEARNING (Random Forest + LOO-CV):")
print(f"  - R² Score: {r2:.4f}")
print(f"  - MAE: {mae:.0f} casos")
print(f"  - RMSE: {rmse:.0f} casos")
print(f"  - Feature más importante: {feature_names[indices[0]]} ({importances[indices[0]]:.4f})")

print(f"\nARCHIVOS GENERADOS:")
print(f"  - Gráficos PNG: 5 (red, distribución, top10, SIR real, comparación, features, predicción)")
print(f"  - CSV de métricas: 3 (nodos, alcaldías, ML data)")
print(f"  - Grafo GraphML: 1 (metro_network)")
print(f"  - Resumen JSON: 1 (resumen_metricas)")

print(f"\nDATOS PROCESADOS:")
print(f"  - Estaciones de metro: {df_metro['estacion'].nunique()}")
print(f"  - Líneas de metro: {df_metro['linea'].nunique()}")
print(f"  - Casos COVID CDMX (2020-2021): {len(df_covid):,}")
print(f"  - Alcaldías procesadas: {len(df_alcaldia_metrics)}")

print("\n" + "=" * 80)
print(f"TIEMPO DE EJECUCIÓN: {datetime.now().isoformat()}")
print("=" * 80)
print("\n✓ PROYECTO COMPLETADO EXITOSAMENTE")
print("  Próximo paso: Generar reporte LaTeX desde resultados_metricas.json")